# Pollution Risk Clustering — Indian Air Quality
**DAV 16 | CSE520 | Ahmedabad University**

Team: Hiir Jadav · Preet Kaur · Aarya Gogia · Vrushank Thakkar

---

**Cities:** Ahmedabad · Chennai · Delhi · Kolkata · Shillong · Mumbai  
**Dataset:** CPCB Air Quality Data via Kaggle (2015–2023, ~1M records)  
**Goal:** Cluster cities by pollution risk using K-Means + Time-Series Decomposition

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from src.preprocess import run_pipeline, get_null_report, POLLUTANTS
from src.clustering import run_clustering
from src.timeseries import (
    plot_yearly_trend, plot_seasonal_pattern,
    plot_season_boxplot, plot_correlation_heatmap, plot_decomposition
)

print('✅ All imports successful')

## 1. Load & Preprocess Data

In [ ]:
# Run the full preprocessing pipeline
# Make sure your CSVs are in the ../data/ folder
hourly_df, monthly_df = run_pipeline(data_dir='../data/')

print(f'\nHourly data shape : {hourly_df.shape}')
print(f'Monthly data shape: {monthly_df.shape}')

## 2. Null Value Report

In [ ]:
null_report = get_null_report(hourly_df)
print('Missing values after preprocessing:\n')
print(null_report.to_string(index=False))

# Visualize null % as a bar chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(null_report['Column'], null_report['Null %'], color='steelblue')
ax.set_xlabel('Null %')
ax.set_title('Missing Value % per Pollutant (after cleaning)')
plt.tight_layout()
plt.savefig('../outputs/null_report.png', dpi=150)
plt.show()

## 3. Exploratory Data Analysis

In [ ]:
# Year-wise PM2.5 trend
fig = plot_yearly_trend(monthly_df, pollutant='PM2.5')
fig.show()

In [ ]:
# Seasonal pattern — month-wise average
fig = plot_seasonal_pattern(monthly_df, pollutant='PM2.5')
fig.show()

In [ ]:
# Boxplot by Indian meteorological seasons
fig = plot_season_boxplot(monthly_df, pollutant='PM2.5')
fig.show()

In [ ]:
# Correlation heatmap for Delhi
fig = plot_correlation_heatmap(monthly_df, city='Delhi')
fig.show()

## 4. K-Means Clustering

In [ ]:
# Run clustering with k=3 (High / Medium / Low risk)
results = run_clustering(monthly_df, k=3)

city_features = results['city_features']
print('\nCity cluster assignments:')
print(city_features[['City', 'Risk_Label', 'PM2.5', 'PM10', 'NO2']].to_string(index=False))

In [ ]:
# Elbow curve
results['fig_elbow'].show()

In [ ]:
# PCA 2D cluster plot
results['fig_clusters_2d'].show()

In [ ]:
# Heatmap of normalized pollutant profiles
results['fig_heatmap'].show()

In [ ]:
# PM2.5 bar chart by city
results['fig_pm25_bar'].show()

## 5. Time-Series Decomposition

In [ ]:
# Decompose Delhi's PM2.5 time series
fig = plot_decomposition(monthly_df, city='Delhi', pollutant='PM2.5')
fig.show()

In [ ]:
# Try another city
fig = plot_decomposition(monthly_df, city='Mumbai', pollutant='PM2.5')
fig.show()

## 6. Save Outputs

In [ ]:
os.makedirs('../outputs', exist_ok=True)

# Save monthly data for Streamlit
monthly_df.to_csv('../data/monthly_city_data.csv', index=False)
city_features.to_csv('../outputs/cluster_results.csv', index=False)

print('✅ Outputs saved to data/ and outputs/ folders')